# Create a Tile-Count Map

A tile-count map (tilegram) assigns each region an exact integer number of tiles.
The tile count is the data: a region with twice the value gets twice as many tiles,
and all tiles are the same size.

This guide covers two parameters that control how regions map to tiles:

- **`tile_count`** — column of integers; each region gets exactly that many tiles
- **`group_by`** — column of labels; each region gets one tile but shares a group
  label for group-level export and per-group styling

These parameters are mutually exclusive.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import carto_flow.symbol_cartogram as sym
from carto_flow.data import load_us_census
from carto_flow.symbol_cartogram.styling import Styling

gdf_state = load_us_census()
gdf_district = load_us_census(level="congressional_district")  # simplify=5000)

In [ ]:
gdf = gdf_state.join(gdf_district.groupby("STATE")["CONGRESSIONAL_DISTRICT"].count(), on="STATE")
gdf.loc[gdf["CONGRESSIONAL_DISTRICT"].isna(), "CONGRESSIONAL_DISTRICT"] = 0

## Compute seat counts

US House seats are apportioned by population using the Hamilton method:
give each state its integer floor, then award the remaining seats to
states with the largest fractional remainders.
The result is a column of positive integers that sum to 435.

In [ ]:
pop = gdf["Population"].values.astype(float)
seats_float = pop / pop.sum() * 435
seats = np.floor(seats_float).astype(int)
# Distribute remaining seats to states with largest remainders
n_remaining = 435 - seats.sum()
top_remainder_idx = np.argsort(seats_float - seats)[::-1][:n_remaining]
seats[top_remainder_idx] += 1

gdf = gdf.copy()
gdf["Seats"] = seats

print(f"Total seats: {gdf['Seats'].sum()}")
gdf[["State Abbreviation", "Population", "Seats"]].sort_values("Seats", ascending=False).head(10)

## Build the tilegram

Pass the seat-count column to `tile_count`.
The grid layout assigns each state exactly `Seats` hexagonal tiles while
preserving geographic adjacency.

In [ ]:
%%time

result = sym.create_symbol_cartogram(
    gdf_district,
    group_by="STATE",
    collapse_group=0.95,
    layout=sym.CirclePackingLayout(
        sym.CirclePackingLayoutOptions(
            group_weight=0.5,
            origin_weight=0.2,
            topology_weight=0,
            compactness=0.0,
            spacing=0.2,
            neighbor_weight=0.1,
            max_iterations=500,
            expansion=0.0,
            advanced=sym.CirclePackingAdvancedOptions(local_step_fraction=0.1),
        )
    ),
    size_normalization="total",
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
gdf.plot(color="none", ax=ax)
_ = result.plot("STATE", ax=ax, legend=False, label="State Abbreviation")
# plt.axis('on');

In [ ]:
layout_result = sym.create_layout(
    gdf,
    tile_count="Seats",
    layout="grid",
)
result = layout_result.style()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
result.plot(
    ax=ax,
    source_gdf=gdf,
    facecolor="Population Density",
    cmap="viridis",
    legend=True,
    legend_kwds={"shrink": 0.6, "label": "Population density"},
)
ax.set_title("US House seats (435 tiles) — shaded by population density")
ax.axis("off")
plt.tight_layout()

## Inspect the result

`result.symbols` has one row per tile.
Two index columns are always present when `tile_count` is used:

- **`original_index`** — row index in `gdf` that this tile belongs to
- **`group_index`** — same as `original_index` for `tile_count` (each state is its own group)

In [ ]:
print(f"{len(result.symbols)} tiles total")
result.symbols[["geometry", "original_index", "group_index", "_symbol_size"]].head(8)

## Export group-level geometries

`to_geodataframe(level="group")` merges all tiles belonging to the same state
into a union polygon and returns one row per state.
Pass `source_gdf` to include the original attribute columns.

In [ ]:
gdf_states = result.to_geodataframe(source_gdf=gdf, level="group")

print(f"{len(gdf_states)} rows (one per state)")
print(f"Columns: {gdf_states.columns.tolist()}")
gdf_states[["State Abbreviation", "Seats", "tile_count"]].sort_values("tile_count", ascending=False).head(10)

The `tile_count` column records how many tiles were actually assigned
(which should equal `Seats` for every state).

## Group-by: per-group symbol shapes

`group_by` labels each region with a group identifier without changing the
number of tiles. The layout still assigns one symbol per region (sized by
`size`), and stores a `group_index` on each tile that can be used for
per-group styling and group-level export.

Whether the grouping also affects *placement* depends on the layout.
`CirclePackingLayout` adds a pull toward the group centroid whose strength is
`group_weight`; at the default of 0.0 the grouping is inert for placement and
the layout warns. Below it is set to 0.5, so states of the same census region
are drawn together as well as sharing a symbol shape.


In [ ]:
layout_result_grp = sym.create_layout(
    gdf,
    size="Population",
    group_by="Region",
    layout=sym.CirclePackingLayout(group_weight=0.5),
)

In [ ]:
# Map region names to the integer group indices assigned by prepare_layout_data
region_to_gid = {
    region: int(gid)
    for region, gid in zip(
        gdf["Region"].values,
        layout_result_grp.group_ids,
        strict=False,
    )
}
print(region_to_gid)

In [ ]:
region_symbols = {
    "Northeast": "circle",
    "Midwest": "square",
    "South": "hexagon",
    "West": "diamond",
}

styling = Styling()
for region, symbol in region_symbols.items():
    styling = styling.set_group_symbol(symbol, group_indices=[region_to_gid[region]])

result_grp = layout_result_grp.style(styling)

In [ ]:
from matplotlib.patches import Patch

# Build a per-symbol color array from the Region column
region_colors = {"Northeast": "#4e79a7", "Midwest": "#f28e2b", "South": "#e15759", "West": "#76b7b2"}
orig_idx = result_grp.symbols["original_index"].values
facecolors = [region_colors[gdf["Region"].iloc[i]] for i in orig_idx]

fig, ax = plt.subplots(figsize=(12, 7))
result_grp.plot(
    ax=ax,
    facecolor=facecolors,
    edgecolor="white",
    linewidth=0.4,
    alpha=0.85,
)


handles = [
    Patch(facecolor=color, label=f"{region} ({region_symbols[region]})", edgecolor="white")
    for region, color in region_colors.items()
]
ax.legend(handles=handles, loc="lower left", framealpha=0.9)
ax.set_title("US states sized by population — symbol shape by census region")
ax.axis("off")
plt.tight_layout()

## Export group-level geometries with `group_by`

`to_geodataframe(level="group")` works the same way here:
one row per region, with the union of all state symbols as geometry.

In [ ]:
gdf_regions = result_grp.to_geodataframe(source_gdf=gdf, level="group")
print(f"{len(gdf_regions)} rows (one per region)")
gdf_regions[["Region", "tile_count"]]

## Choosing a layout for tile-count maps

The `"grid"` layout (used above) is fast and works well for most tilegrams.
It treats each tile-count value as a symbol size and places one hexagon per
region, so topology preservation is its main objective.

The `"mosaic"` layout is designed specifically for tile-count maps:
it assigns each region *exactly* the right number of tiles and uses a
flow-morphing pre-step to improve geographic placement.
It is slower but produces more accurate tilegrams for large count variations:

In [ ]:
layout_mosaic = sym.create_layout(
    gdf,
    tile_count="Seats",
    layout="mosaic",
)
result_mosaic = layout_mosaic.style()

fig, ax = plt.subplots(figsize=(12, 7))
result_mosaic.plot(
    ax=ax,
    source_gdf=gdf,
    facecolor="Region",
    cmap="Set2",
    legend=True,
)
ax.set_title("US House seats — mosaic layout, shaded by region")
ax.axis("off")
plt.tight_layout()